In [18]:
import pandas as pd

path = "data/yellow_tripdata_2025-03.parquet"

# pick a sample of the data to perform EDA
df = pd.read_parquet(path).head(1000000)

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-03-01 00:17:16,2025-03-01 00:25:52,1.0,0.90,1.0,N,140,236,1,7.9,3.50,0.5,2.60,0.0,1.0,15.50,2.5,0.0,0.00
1,1,2025-03-01 00:37:38,2025-03-01 00:43:51,1.0,0.60,1.0,N,140,262,1,6.5,3.50,0.5,2.30,0.0,1.0,13.80,2.5,0.0,0.00
2,2,2025-03-01 00:24:35,2025-03-01 00:39:49,1.0,1.94,1.0,N,161,68,1,14.9,1.00,0.5,5.16,0.0,1.0,25.81,2.5,0.0,0.75
3,2,2025-03-01 00:56:16,2025-03-01 01:01:35,2.0,0.95,1.0,N,231,13,1,7.2,1.00,0.5,2.59,0.0,1.0,15.54,2.5,0.0,0.75
4,1,2025-03-01 00:01:44,2025-03-01 00:10:00,1.0,1.50,1.0,N,163,236,1,8.6,4.25,0.5,2.85,0.0,1.0,17.20,2.5,0.0,0.75


In [19]:

# Basic info
print("=== BASIC INFO ===")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n=== DATA TYPES ===")
print(df.dtypes)
print("\n=== NULL COUNTS ===")
print(df.isnull().sum())
print("\n=== DESCRIBE ===")
print(df.describe())

=== BASIC INFO ===
Rows: 1,000,000
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
Memory: 181.20 MB

=== DATA TYPES ===
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amo

In [44]:
# 1. Trip distance anomali
print(f"Trip distance = 0: {len(df[df['trip_distance'] == 0]):,}")
print(f"Trip distance < 0: {len(df[df['trip_distance'] < 0]):,}")
print(f"Trip distance > 100 (outlier): {len(df[df['trip_distance'] > 100]):,}")
print("==========================================")

# 2. Fare amount anomali
print(f"Fare amount <= 0: {len(df[df['fare_amount'] <= 0]):,}")
print(f"Fare amount > 500 (outlier): {len(df[df['fare_amount'] > 500]):,}")
print("==========================================")

# 3. Passenger count anomali
print(f"Passenger count = 0: {len(df[df['passenger_count'] == 0]):,}")
print(f"Passenger count < 0: {len(df[df['passenger_count'] < 0]):,}")
print(f"Passenger count > 6: {len(df[df['passenger_count'] > 6]):,}")
print("==========================================")

# 4. Rate code ID invalid (valid: 1-6)
valid_rate = [1, 2, 3, 4, 5, 6, 99]  # 99 = unknown
invalid_rate = df[~df['RatecodeID'].isin(valid_rate)]
print(f"Invalid RatecodeID: {len(invalid_rate):,}")
print("==========================================")

# 5. Payment type invalid (valid: 1-5)
valid_payment = [1,2,3,4,5,6]  # 1=Credit, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided
invalid_payment = df[~df['payment_type'].isin(valid_payment)]
print(f"Invalid payment_type: {len(invalid_payment):,}")
print("==========================================")

Trip distance = 0: 13,136
Trip distance < 0: 0
Trip distance > 100 (outlier): 20
Fare amount <= 0: 21,314
Fare amount > 500 (outlier): 21
Passenger count = 0: 7,061
Passenger count < 0: 0
Passenger count > 6: 7
Invalid RatecodeID: 0
Invalid payment_type: 0


In [48]:
# 6. Duration anomali (trip too long > 24 hours?)
df['duration_hours'] = (pd.to_datetime(df['tpep_dropoff_datetime']) - 
                         pd.to_datetime(df['tpep_pickup_datetime'])).dt.total_seconds() / 3600
print(f"Duration > 24 hours: {len(df[df['duration_hours'] > 24]):,}")

# ============================================================
# 7. VendorID & store_and_fwd_flag
# ============================================================
print("=== VENDOR ID ===")
valid_vendor = [1, 2, 6, 7]
invalid_vendor = df[~df['VendorID'].isin(valid_vendor)]
print(f"VendorID value counts:\n{df['VendorID'].value_counts().sort_index()}")
print(f"\nInvalid VendorID (not in {valid_vendor}): {len(invalid_vendor):,}")
print("==========================================")

print("=== STORE & FORWARD FLAG ===")
valid_sff = ['Y', 'N']
print(f"store_and_fwd_flag value counts:\n{df['store_and_fwd_flag'].value_counts()}")
invalid_sff = df[~df['store_and_fwd_flag'].isin(valid_sff)]
print(f"Invalid store_and_fwd_flag: {len(invalid_sff):,}")
print("==========================================")

Duration > 24 hours: 4
=== VENDOR ID ===
VendorID value counts:
VendorID
1    222071
2    774199
7      3730
Name: count, dtype: int64

Invalid VendorID (not in [1, 2, 6, 7]): 0
=== STORE & FORWARD FLAG ===
store_and_fwd_flag value counts:
store_and_fwd_flag
N    997298
Y      2702
Name: count, dtype: int64
Invalid store_and_fwd_flag: 0


In [51]:
# ============================================================
# 8. extra & mta_tax
# ============================================================
print("=== EXTRA ===")
print(f"Extra < 0: {len(df[df['extra'] < 0]):,}")
print(f"Extra > 10: {len(df[df['extra'] > 10]):,}")
print("==========================================")

print("=== MTA TAX ===")
# Normalnya $0.50 untuk sebagian besar trip
print(f"MTA tax != 0.50: {len(df[df['mta_tax'] != 0.50]):,}")
print(f"MTA tax < 0: {len(df[df['mta_tax'] < 0]):,}")
print(f"MTA tax value counts:\n{df['mta_tax'].value_counts().sort_index()}")
print("==========================================")

# ============================================================
# 9. tip_amount & tolls_amount
# ============================================================
print("=== TIP AMOUNT ===")
print(f"Tip < 0: {len(df[df['tip_amount'] < 0]):,}")
print(f"Tip > fare_amount: {len(df[df['tip_amount'] > df['fare_amount']]):,}")
# Tip tercatat > 0 padahal payment Cash (tip tunai tidak tercatat di sistem)
cash_tip = df[(df['payment_type'] == 2) & (df['tip_amount'] > 0)]
print(f"Tip > 0 with Cash payment (anomali): {len(cash_tip):,}")
print("==========================================")

print("=== TOLLS AMOUNT ===")
print(f"Tolls < 0: {len(df[df['tolls_amount'] < 0]):,}")
print("==========================================")

=== EXTRA ===
Extra < 0: 10,153
Extra > 10: 1,941
=== MTA TAX ===
MTA tax != 0.50: 33,289
MTA tax < 0: 20,033
MTA tax value counts:
mta_tax
-0.50     20033
 0.00     13255
 0.50    966711
 1.32         1
Name: count, dtype: int64
=== TIP AMOUNT ===
Tip < 0: 38
Tip > fare_amount: 22,197
Tip > 0 with Cash payment (anomali): 64
=== TOLLS AMOUNT ===
Tolls < 0: 1,716


In [53]:
# ============================================================
# 10. improvement_surcharge, congestion_surcharge & airport_fee
# ============================================================
print("=== IMPROVEMENT SURCHARGE ===")
# Normalnya $0.30 sejak 2015
print(f"Improvement surcharge != 0.30: {len(df[df['improvement_surcharge'] != 0.30]):,}")
print(f"Improvement surcharge value counts:\n{df['improvement_surcharge'].value_counts().sort_index()}")
print("==========================================")

print("=== CONGESTION SURCHARGE ===")
# Normal: $2.50 Yellow Taxi, $1.25 Green Taxi (Manhattan), $0 di luar zona
print(f"Congestion surcharge < 0: {len(df[df['congestion_surcharge'] < 0]):,}")
print(f"Congestion surcharge value counts:\n{df['congestion_surcharge'].value_counts().sort_index()}")
print("==========================================")

print("=== AIRPORT FEE ===")
# Normal: $1.25 LaGuardia, $2.50 JFK, $0 non-airport
print(f"Airport fee < 0: {len(df[df['Airport_fee'] < 0]):,}")
print(f"Airport fee value counts:\n{df['Airport_fee'].value_counts().sort_index()}")
print("==========================================")

=== IMPROVEMENT SURCHARGE ===
Improvement surcharge != 0.30: 1,000,000
Improvement surcharge value counts:
improvement_surcharge
-1.0     20845
 0.0     12373
 1.0    966782
Name: count, dtype: int64
=== CONGESTION SURCHARGE ===
Congestion surcharge < 0: 16,825
Congestion surcharge value counts:
congestion_surcharge
-2.5     16825
 0.0     76584
 2.5    906591
Name: count, dtype: int64
=== AIRPORT FEE ===
Airport fee < 0: 3,837
Airport fee value counts:
Airport_fee
-1.75      3837
 0.00    920183
 1.75     75888
 5.00        25
 6.75        67
Name: count, dtype: int64


In [54]:
# ============================================================
# 11. Duration negatif, PULocationID/DOLocationID, total_amount integrity
# ============================================================
print("=== DURATION NEGATIF ===")
print(f"Duration < 0 (dropoff before pickup): {len(df[df['duration_hours'] < 0]):,}")
print("==========================================")

=== DURATION NEGATIF ===
Duration < 0 (dropoff before pickup): 2


In [56]:
print("=== PULocationID / DOLocationID ===")
# Range zona TLC umumnya 1-265
print(f"PULocationID <= 0: {len(df[df['PULocationID'] <= 0]):,}")
print(f"DOLocationID <= 0: {len(df[df['DOLocationID'] <= 0]):,}")
print(f"PULocationID > 265: {len(df[df['PULocationID'] > 265]):,}")
print(f"DOLocationID > 265: {len(df[df['DOLocationID'] > 265]):,}")
print("==========================================")

print("=== TOTAL AMOUNT INTEGRITY ===")
# total_amount = fare_amount + extra + mta_tax + tip_amount + tolls_amount 
#              + improvement_surcharge + congestion_surcharge + Airport_fee
calc_total = (
    df['fare_amount'] + df['extra'] + df['mta_tax'] + df['tip_amount'] +
    df['tolls_amount'] + df['improvement_surcharge'] + 
    df['congestion_surcharge'] + df['Airport_fee']
)
mismatch = ~(df['total_amount'].round(2) == calc_total.round(2))
print(f"Total amount mismatch (tidak sesuai komponen): {mismatch.sum():,}")
if mismatch.sum() > 0:
    print("Sample mismatch:")
    print(df[mismatch][['fare_amount', 'extra', 'mta_tax', 'tip_amount', 
                        'tolls_amount', 'improvement_surcharge',
                        'congestion_surcharge', 'Airport_fee', 'total_amount']].head(5))
print("==========================================")

=== PULocationID / DOLocationID ===
PULocationID <= 0: 0
DOLocationID <= 0: 0
PULocationID > 265: 0
DOLocationID > 265: 0
=== TOTAL AMOUNT INTEGRITY ===
Total amount mismatch (tidak sesuai komponen): 787,054
Sample mismatch:
   fare_amount  extra  mta_tax  tip_amount  tolls_amount  \
0          7.9   3.50      0.5        2.60           0.0   
1          6.5   3.50      0.5        2.30           0.0   
2         14.9   1.00      0.5        5.16           0.0   
3          7.2   1.00      0.5        2.59           0.0   
4          8.6   4.25      0.5        2.85           0.0   

   improvement_surcharge  congestion_surcharge  Airport_fee  total_amount  
0                    1.0                   2.5          0.0         15.50  
1                    1.0                   2.5          0.0         13.80  
2                    1.0                   2.5          0.0         25.81  
3                    1.0                   2.5          0.0         15.54  
4                    1.0          

In [38]:
df['PULocationID'].head()

0    140
1    140
2    161
3    231
4    163
Name: PULocationID, dtype: int32

In [25]:
df['VendorID'].value_counts()

VendorID
2    774199
1    222071
7      3730
Name: count, dtype: int64